# Long Read Simulation
We are interested in comparing linked-read inversion-calling performance to long-read inversion calling performance. Since the price of long reads varies between vendors and the price will change dramatically over the course of this project and the time thereafter, we will restrict the long-read simulations to representatives of:
- all samples
- all inversion size classes
- only depths `0.5`, `5`, `20`

## The Workflow
During the pilot runs of this project, we were able to make a Snakemake workflow of this process, which was modified as the needs of this work became clearer. As such, the Snakefile is provided below in its entirety. However, most of the code is workflow-specific nuance and the most relevant parts of the workflow are annotated below.

The workflow was invoked using:
```bash
snakemake -s longread_workflow/workflow/Snakefile --cores 12 --directory longread_workflow --sdm conda
```

In [ ]:
reference_genome = "../dmel_genome/dmel_2_3.fa.gz"
samples = ["sample_%02d" % i for i in range(1,11)]
sizes = ["small", "medium", "large", "xl"]
depths = ["05", "5", "20"]
platforms = ["pacbio", "ontshort", "ontlong"]
modes = {
    "pacbio" : "--difference-ratio 22:45:33 --length-mean 8000 --length-sd 1000 --length-min 2000 --length-max 20000",
    "ontshort" : "--difference-ratio 39:24:36 --length-mean 3000 --length-sd 1000",
    "ontlong" : "--difference-ratio 39:24:36 --length-mean 50000 --length-sd 20000"
    }

wildcard_constraints:
    runtype = "[a-zA-Z0-9]+",
    size = "[a-zA-Z0-9]+",
    depths = "[0-9]+"

rule all:
    default_target: True
    input:
        expand("sv/{sample}.{runtype}.{size}.{depth}.bcf", sample = samples, runtype =platforms, size = sizes, depth = depths),
        expand("logs/{sample}.{runtype}.{size}.{depth}.lengthdist", sample = samples, runtype =platforms, size = sizes, depth = depths)
    message:
        "Complete!"

rule decompress_fasta:
    input: "../simulated_variants/inversions_snps_genomes/{size}/{sample}.snp_inv.hap{hap}.fasta.gz"
    output: temp("fasta/{size}.{sample}.snp_inv.hap{hap}.fa")
    message: "Temporarily decompressing {input}"
    shell: "gzip -dc {input} > {output}"

rule simluate_longreads:
    input:
        geno = "fasta/{size}.{sample}.snp_inv.hap{hap}.fa",
        model = "models/QSHMM-ONT-HQ.model"
    output:
        temp(expand("{{sample}}.{{runtype}}.{{size}}.{{depth}}.{{hap}}_{chunk}.fastq", chunk = ["%04d" % i for i in range(1,8)]))
    log:
        "logs/{sample}.{runtype}.{size}.{depth}.{hap}.log"
    params:
        config = lambda wc: modes[wc.get("runtype")],
        depth = lambda wc: "--depth " + wc.get("depth"),
        prefix = lambda wc: ".".join([wc.get(i) for i in ["sample", "runtype", "size", "depth", "hap"]])
    message:
        "Simulating reads: {wildcards.sample} {wildcards.runtype}.{wildcards.size}.{wildcards.depth}.haplotype_{wildcards.hap}"
    conda:
        "envs/longreads.yaml"
    shell:
        """
        pbsim --strategy wgs --method qshmm --seed 6969 --qshmm {input.model} \\
            --genome {output.geno} {params.depth} {params.config} \\
            --prefix {params.prefix} --id-prefix {params.prefix} 2> {log} &&
            rm {params.prefix}*.maf {params.prefix}*.ref         
        """

rule merge_fastq:
    input:
        expand("{{sample}}.{{runtype}}.{{size}}.{{depth}}.{hap}_{chunk}.fastq", hap = [1,2], chunk = ["%04d" % i for i in range(1,8)])
    output:
        "{sample}.{runtype}.{size}.{depth}.fq"
    params:
        lambda wc: modes[wc.get("runtype")]
    message:
        "Concatenating fastq: {wildcards.sample} {wildcards.runtype}.{wildcards.size}.{wildcards.depth}"
    shell:
        "cat {input} > {output}"

rule compress:
    input:
        "{sample}.{runtype}.{size}.{depth}.fq"
    output:
        "{sample}.{runtype}.{size}.{depth}.fq.gz"
    threads:
        6
    conda:
        "envs/longreads.yaml"
    message:
        "Compressing: {input}"
    shell:
        "pigz -p {threads} {input}"

rule length_distribution:
    input:
        "{sample}.{runtype}.{size}.{depth}.fq.gz"
    output:
        "logs/{sample}.{runtype}.{size}.{depth}.lengthdist"
    message:
        "Creating read length histogram: {wildcards.sample} {wildcards.runtype}.{wildcards.size}.{wildcards.depth}"
    shell:
        """
        awk 'NR%4 == 2 {{lengths[length($0)]++}} END {{for (l in lengths) {{print l, lengths[l]}}}}' <(zcat {input}) > {output}
        """

rule align:
    input:
        geno = reference_genome,
        reads = "{sample}.{runtype}.{size}.{depth}.fq.gz"
    output:
        temp("align/{sample}.{runtype}.{size}.{depth}.sam")
    log:
        "logs/{sample}.{runtype}.{size}.{depth}.align.log"
    threads:
        6
    params:
        lambda wc: "map-ont" if "ont" in wc.get("runtype") else "map-pb" 
    message:
        "Aligning: {wildcards.sample} {wildcards.runtype}.{wildcards.size}.{wildcards.depth} in {params} mode"
    conda:
        "envs/longreads.yaml"
    shell:
        "minimap2 -t {threads} -ax {params} {input} > {output} 2> {log}"

rule filter:
    input:
        "align/{sample}.{runtype}.{size}.{depth}.sam"
    output:
        bam = temp("align/{sample}.{runtype}.{size}.{depth}.filt.bam"),
        bai = temp("align/{sample}.{runtype}.{size}.{depth}.filt.bam.bai")
    log:
        "logs/{sample}.{runtype}.{size}.{depth}.filter.log"
    threads:
        2
    message:
        "Filtering and sorting:  {wildcards.sample} {wildcards.runtype}.{wildcards.size}.{wildcards.depth}"
    conda:
        "envs/longreads.yaml"
    shell:
        """
        samtools view -h -F 4 -q 20 {input} | 
            samtools sort -O bam -l 0 -m 4G --write-index -o {output.bam}##idx##{output.bai} 2> {log}
        """

rule mark_duplicates:
    input:
        bam = "align/{sample}.{runtype}.{size}.{depth}.filt.bam",
        bai = "align/{sample}.{runtype}.{size}.{depth}.filt.bam.bai"
    output:
        bam = "align/{sample}.{runtype}.{size}.{depth}.bam",
        bai = "align/{sample}.{runtype}.{size}.{depth}.bam.bai"
    log:
        "logs/{sample}.{runtype}.{size}.{depth}.markduplicates.log"
    threads:
        4
    message:
        "Marking duplicates:  {wildcards.sample} {wildcards.runtype}.{wildcards.size}.{wildcards.depth}"
    conda:
        "envs/longreads.yaml"
    shell:
        "sambamba markdup -t {threads} -l 0 {input.bam} {output.bam} 2> {log}"

rule call_variants:
    input:
        gen = reference_genome,
        reads = "align/{sample}.{runtype}.{size}.{depth}.bam"
    output:
        "sv/{sample}.{runtype}.{size}.{depth}.bcf"
    params:
        lambda wc: "ont" if "ont" in wc.get("runtype") else "pb" 
    message:
        "Calling variants: {wildcards.sample} {wildcards.runtype}.{wildcards.size}.{wildcards.depth} with mode {params}"
    conda:
        "envs/longreads.yaml"
    shell:
        "delly lr -y {params} -o {output} -g {input}"


### Housekeeping
#### The `modes`
There is a dict at the top that sets configurations for PacBio and Oxford Nanopore simulations with `pbsim3`. It specifies that we want the PacBio data to have an average read length of 8kpb (sd = 1kbp) with min/max of 2kbp/20kbp. There are likewise two nanopore datesets (`ontshort` and `ontlong`), where the short has a mean length of 3kbp (sd = 1kbp) and the long has a mean of 50kbp (sd = 20kbp).
```python
modes = {
    "pacbio" : "--difference-ratio 22:45:33 --length-mean 8000 --length-sd 1000 --length-min 2000 --length-max 20000",
    "ontshort" : "--difference-ratio 39:24:36 --length-mean 3000 --length-sd 1000",
    "ontlong" : "--difference-ratio 39:24:36 --length-mean 50000 --length-sd 20000"
    }
```

#### Decompression
`pbsim3` doesn't play nice with gzipped fasta files, so they need to be decompressed:
```bash
gzip -dc {input} > {output}
```

### Simulate Long Reads
Using `pbsim3`, we are simulating long reads from the variant-modified genomes for all samples. This rule iterates over samples, inversion size classes, depths, and the `pacbio`, `ontshort`, and `ontlong` platforms.
```bash
pbsim --strategy wgs --method qshmm --seed 6969 --qshmm {model} \
    --genome {reference_genome} {depth} {mode} \
    --prefix {prefix} --id-prefix {params.prefix} 2> {log} &&
    rm {params.prefix}*.maf.gz {params.prefix}*.ref         
```

### Length Distribution
This is something of a sanity check, but here we are getting the distribution of read lengths.
```bash
awk 'NR%4 == 2 {lengths[length($0)]++} END {for (l in lengths) {print l, lengths[l]}}' <(zcat {input}) > {output}
```

### Aligning Reads
Like most variant-calling approaches, we first need to align the reads to the reference genome. We use the long-read mapper `minimap2` for that and do some basic alignment filtering.
```bash
minimap2 -t {threads} -ax {params} {input} > {output} 2> {log}

samtools view -h -F 4 -q 20 {input} | 
    samtools sort -O bam -l 0 -m 4G --write-index -o {output.bam}##idx##{output.bai} 2> {log}
```

### Marking Duplicates
As a precaution, we need to mark duplicates in the reads, if there are any.
```bash
sambamba markdup -t {threads} -l 0 {input.bam} {output.bam} 2> {log}
```

### Call Structural Variants
Finally, we call variants using `delly`.
```bash
delly lr -y {params} -o {output} -g {input}
```